# 12 — Build real-cohort v2 features

Standardise LBC GSE178936, GSE288742 and SardiNIA; build clone-dynamics features; reuse existing BioNeMo/ESM-2 residue-delta vectors. **This is measurement validation, not leukemia-progression training.**

In [8]:
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection"
)

os.chdir(PROJECT)

SRC = PROJECT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
from pathlib import Path
from real_data_adapter import *
from clone_dynamics import patient_dynamics_features
import pandas as pd, numpy as np, json

In [11]:
cfg = RealDataConfig(min_vaf=0.01, min_depth_lbc=500, min_timepoints=3)
raw = Path(PROJECT)/'data'/'raw'
processed = Path(PROJECT)/'data'/'processed'
parts=[]
parts.append(qc_and_filter_serial(load_lbc_serial(raw/'lbc_gse178936'/'GSE178936_LBC_ARCHER.1PCT_VAF.Feb22.non-synonymous.tsv','GSE178936',cfg),cfg))
parts.append(qc_and_filter_serial(load_lbc_serial(raw/'gse288742'/'GSE288742_ELC_CHIP.2024_LBC_Cohort.1PCT_VAF_NON-SYNONYMOUS.tsv','GSE288742',cfg),cfg))
parts.append(qc_and_filter_serial(load_sardinia_serial(raw/'sardinia'/'41586_2022_4785_MOESM5_ESM.xlsx',cfg),cfg))
serial=pd.concat(parts,ignore_index=True,sort=False)
print(audit_serial(serial))

{'rows': 1483, 'patients': 265, 'variants': 423, 'missense_rows': 818, 'cohorts': {'GSE178936': {'rows': 434, 'patients': 57, 'variants': 136, 'median_timepoints_per_variant': 3.0, 'median_span_years': 6.0}, 'GSE288742': {'rows': 114, 'patients': 28, 'variants': 38, 'median_timepoints_per_variant': 3.0, 'median_span_years': 12.0}, 'SardiNIA': {'rows': 935, 'patients': 180, 'variants': 249, 'median_timepoints_per_variant': 4.0, 'median_span_years': 10.309999999999995}}}


In [12]:
patients=latest_patient_table(serial)
dynamics=patient_dynamics_features(serial[['patient_id','variant_id','time_years','vaf']])
emb=pd.read_pickle(processed/'lbc_sardinia_with_delta.pkl')
protein,coverage=bionemo_patient_matrix(serial,emb,patients.patient_id.tolist())
features=patients.merge(dynamics,on='patient_id',how='left').merge(coverage,on='patient_id',how='left')
features.head()

,patient_id,cohort,n_mutations,max_vaf,mean_vaf,sum_vaf,n_genes,genes,mut_keys,age,...,slope_dispersion,max_observed_vaf,median_trajectory_rmse,n_candidate_coclone_pairs,mean_pairwise_log_vaf_corr,structure_identifiable,n_mutations_at_index,n_bionemo_mutations,bionemo_coverage,has_embedding
0,CHIP_LBC21_002,GSE178936,1,0.0149,0.014900,0.0149,1,[DNMT3A],[DNMT3A_I769S],NaN,...,0.000000,0.0221,0.149196,0,NaN,False,1,1,1.000000,1
1,CHIP_LBC21_004,GSE178936,1,0.0146,0.014600,0.0146,1,[RAD21],[RAD21_D400A],NaN,...,0.000000,0.0238,0.150282,0,NaN,False,1,1,1.000000,1
2,CHIP_LBC21_006,GSE178936,3,0.0155,0.012067,0.0362,3,"[GATA2, LUC7L2, ZRSR2]","[GATA2_T457P, LUC7L2_nan, ZRSR2_D386A]",NaN,...,0.054156,0.0155,0.025591,1,-0.333858,True,3,2,0.666667,1
3,CHIP_LBC21_009,GSE178936,1,0.0102,0.010200,0.0102,1,[GATA2],[GATA2_T457P],NaN,...,0.000000,0.0124,0.001946,0,NaN,False,1,1,1.000000,1
4,CHIP_LBC21_011,GSE178936,1,0.0112,0.011200,0.0112,1,[BCORL1],[BCORL1_D1198A],NaN,...,0.000000,0.0115,0.041127,0,NaN,False,1,1,1.000000,1


In [13]:
out=processed/'v2'
out.mkdir(parents=True,exist_ok=True)
serial.to_csv(out/'real_serial_filtered.csv',index=False)
features.to_csv(out/'real_patient_features.csv',index=False)
np.save(out/'real_patient_bionemo.npy',protein.astype('float32'))
np.save(out/'real_patient_ids.npy',patients.patient_id.to_numpy(dtype=str))
print('patients',len(features),'protein',protein.shape,'BioNeMo patients',int(coverage.has_embedding.sum()))

patients 265 protein (265, 1280) BioNeMo patients 174


## Important

Do **not** create a synthetic progression label from gene/VAF to train the final model. The next supervised step needs an outcome-linked pre-diagnostic cohort. These assets are the real Stage-B feature layer that will plug into `10_train_fusion.ipynb` once such labels are available.